[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S05/S05_06_crud_python.ipynb)

# S05_06 · CRUD desde Python: `sqlite3` y `pymongo` / `mongomock`
**Sprint 5 · M6b · CRUD desde Python · ⏱ 60 min**

## Qué vas a aprender
1. A conectar Python con una base **SQLite** y a ejecutar consultas con **parámetros** (seguras).
2. A hacer **CRUD** (crear, leer, actualizar, borrar) con `sqlite3`, con `commit` y `rollback`.
3. A **cargar un CSV en la base** y a consultarla con pandas.
4. A repetir el CRUD en **MongoDB** con `pymongo` (aquí, la versión en memoria `mongomock`).

## Contexto TurisData
Ya sabes SQL (Sprint 3) y Python. Ahora unimos ambos: un programa que lee y escribe en la base de datos de
TurisData sin abrir ninguna herramienta gráfica. Además dejaremos la serie diaria de ocupación **dentro de la
base**, que es parte del reto del sprint. Trabajamos sobre una **copia** de `turisdata.db` para no modificar el original.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
import shutil
import sqlite3
import tempfile

import pandas as pd

SEMILLA = 42

# Copia de trabajo de la base de datos (así puedes equivocarte sin miedo)
carpeta = tempfile.mkdtemp()
ruta_bd = os.path.join(carpeta, "turisdata_copia.db")
shutil.copy(dato("turisdata.db"), ruta_bd)
print("Base de trabajo:", ruta_bd)

## 1 · Conectar y leer (R de CRUD)
`sqlite3.connect(ruta)` abre la base. `con.execute("SELECT ...")` ejecuta y devuelve un cursor; `fetchall()` trae
todas las filas (tuplas) y `fetchone()` una. Con `pandas.read_sql_query` obtienes directamente un DataFrame.
Cierra la conexión al acabar (o usa `with`).

In [ ]:
con = sqlite3.connect(ruta_bd)
print(con.execute("SELECT COUNT(*) FROM reservas").fetchone())
print(con.execute("SELECT id_alojamiento, nombre, isla FROM alojamientos LIMIT 3").fetchall())

pd.read_sql_query("SELECT canal, COUNT(*) AS reservas FROM reservas GROUP BY canal", con)

### Ejercicio 1 · Leer con pandas
Guarda en `n_reservas` el número de reservas (con `fetchone()[0]`) y en `tasa_por_isla` un DataFrame con dos
columnas, `isla` y `pct_cancelacion` (tasa de cancelación en % con 1 decimal), calculado con una consulta SQL con
`JOIN` y `GROUP BY` (como en el Sprint 3) y `pd.read_sql_query`. Ordénalo por isla.

In [ ]:
# TODO: SELECT COUNT(*) FROM reservas; y para la tasa: JOIN reservas-alojamientos, ROUND(100.0 * AVG(cancelada), 1), GROUP BY isla ORDER BY isla
...
assert n_reservas == 6000, f"Hay 6000 reservas y tienes {n_reservas}"
assert list(tasa_por_isla.columns) == ["isla", "pct_cancelacion"], f"Las columnas deben ser isla y pct_cancelacion y son {list(tasa_por_isla.columns)}"
assert len(tasa_por_isla) == 6, "Debe haber una fila por isla (6)"
print("Correcto")
print(tasa_por_isla)

## 2 · Consultas con parámetros (¡nunca pegues texto en el SQL!)
Para meter un valor variable en una consulta usa **marcadores `?`** y pasa los valores en una tupla. Así SQLite
escapa el valor y evitas la **inyección SQL** (un ataque clásico: que alguien escriba código SQL en un campo de texto).

```python
con.execute("SELECT ... WHERE id_cliente = ?", (id_cliente,))     # ✅ seguro
con.execute(f"SELECT ... WHERE id_cliente = {id_cliente}")         # ❌ peligroso
```

### Ejercicio 2 · Función de consulta parametrizada
Escribe `reservas_de_cliente(con, id_cliente)` que devuelva la **lista de `id_reserva`** (enteros, ordenados) de ese
cliente, usando un marcador `?`.

In [ ]:
# TODO: con.execute("... WHERE id_cliente = ? ORDER BY id_reserva", (id_cliente,)).fetchall(); saca el primer elemento de cada tupla
...
assert reservas_de_cliente(con, 1234) == [5046, 5760], f"El cliente 1234 tiene las reservas 5046 y 5760 y tienes {reservas_de_cliente(con, 1234)}"
assert reservas_de_cliente(con, 999999) == [], "Un cliente inexistente devuelve una lista vacía"
assert reservas_de_cliente(con, "1 OR 1=1") == [], "Con parámetros, un intento de inyección no devuelve nada"
print("Correcto")

## 3 · Crear (C): insertar filas y cargar un CSV en la base
- Una fila: `con.execute("INSERT INTO ... VALUES (?, ?)", (a, b))`.
- Muchas: `con.executemany(sql, lista_de_tuplas)`.
- **Los cambios no son definitivos hasta `con.commit()`**; `con.rollback()` los deshace.
- Un DataFrame entero: `df.to_sql("tabla", con, if_exists="replace", index=False)`.

In [ ]:
con.execute("INSERT INTO alojamientos (id_alojamiento, nombre, isla, categoria, plazas) VALUES (?, ?, ?, ?, ?)",
            (21, "Alojamiento 21", "El Hierro", "casa rural", 12))
con.commit()
print(con.execute("SELECT * FROM alojamientos WHERE id_alojamiento = 21").fetchall())

### Ejercicio 3 · Poblar la base con la serie diaria
Lee `ocupacion_diaria.csv` con pandas (guárdalo en `diaria`; la fecha como texto) y **cárgalo en una tabla nueva
`ocupacion_diaria`** de la base con `to_sql` (reemplazando si existe). Después guarda en `n_dias` el número de filas de
esa tabla leído **desde la base**.

In [ ]:
# TODO: pd.read_csv(dato("ocupacion_diaria.csv")); diaria.to_sql("ocupacion_diaria", con, if_exists="replace", index=False); SELECT COUNT(*)
...
assert n_dias == 1096, f"La tabla ocupacion_diaria debe tener 1096 filas y tiene {n_dias}"
columnas = [c[1] for c in con.execute("PRAGMA table_info(ocupacion_diaria)").fetchall()]
assert columnas == ["fecha", "ocupacion_pct", "temperatura_c", "festivo", "precio_medio"], f"Columnas inesperadas: {columnas}"
print("Correcto: la base ya contiene la serie diaria")

## 4 · Actualizar (U) y borrar (D), con transacciones
`cursor.rowcount` dice **cuántas filas** ha afectado un `UPDATE` o `DELETE`: úsalo para comprobar que has tocado lo que querías.
Una **transacción** agrupa varias operaciones: o se confirman todas (`commit`) o ninguna (`rollback`).

In [ ]:
cur = con.execute("UPDATE alojamientos SET plazas = plazas + 2 WHERE id_alojamiento = ?", (21,))
print("Filas modificadas:", cur.rowcount)
con.commit()

### Ejercicio 4 · Actualizar y probar un `rollback`
1. Marca como cancelada (`cancelada = 1`) la reserva `5046` con un `UPDATE` con parámetro y guarda en
   `filas_actualizadas` el `rowcount`. Confirma con `commit`.
2. Borra **todas las reservas de 12 noches o más** *dentro de una transacción*, guarda en `borradas` cuántas has borrado y
   luego **deshaz** con `rollback`. Guarda en `total_tras_rollback` el número de reservas que quedan (debe seguir siendo 6000).

In [ ]:
# TODO: UPDATE ... WHERE id_reserva = ? con (5046,), commit; DELETE ... WHERE noches >= 12, rowcount, rollback y COUNT(*)
...
assert filas_actualizadas == 1, f"Debe actualizarse exactamente 1 fila y tienes {filas_actualizadas}"
assert con.execute("SELECT cancelada FROM reservas WHERE id_reserva = 5046").fetchone()[0] == 1, "La reserva 5046 debe quedar cancelada tras el commit"
assert borradas > 0, "Debe haber reservas de 12 noches o más que borrar"
assert total_tras_rollback == 6000, f"Tras el rollback siguen las 6000 reservas y tienes {total_tras_rollback}"
print(f"Correcto: se habrían borrado {borradas} reservas, pero el rollback las devolvió (quedan {total_tras_rollback})")

## 5 · Consultar la base para analizar (de SQL a pandas)
Una vez en la base, podemos combinar SQL para agregar y pandas para analizar o dibujar.

### Ejercicio 5 · Ocupación media por mes desde la base
Con una consulta sobre la tabla `ocupacion_diaria` (`SUBSTR(fecha, 6, 2)` da el mes, como texto `'01'`...`'12'`),
crea `ocupacion_mes`: DataFrame con columnas `mes` y `media` (media redondeada a 2 decimales), ordenado por mes.

In [ ]:
# TODO: SELECT SUBSTR(fecha, 6, 2) AS mes, ROUND(AVG(ocupacion_pct), 2) AS media FROM ocupacion_diaria GROUP BY mes ORDER BY mes
...
assert len(ocupacion_mes) == 12, "Debe haber una fila por mes (12)"
assert ocupacion_mes.loc[0, "mes"] == "01" and ocupacion_mes.loc[0, "media"] == 78.83, "Enero debe tener una media de 78.83"
assert ocupacion_mes.loc[8, "media"] == 60.06, "Septiembre debe tener una media de 60.06"
print("Correcto")
print(ocupacion_mes.T)

In [ ]:
con.close()      # cerrar la conexión al terminar con SQLite

## 6 · El mismo CRUD en MongoDB (`pymongo`)
Con `pymongo` el CRUD es análogo: `insert_one/insert_many`, `find`, `update_one/update_many`, `delete_one/delete_many`.
Aquí usamos `mongomock` (MongoDB en memoria, misma sintaxis) para no necesitar servidor. Con uno real, solo cambia la línea
del cliente: `MongoClient("mongodb://...")`. (Amplía el laboratorio de NoSQL del Sprint 3.)

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("mongomock") is None:      # solo hace falta en Colab (necesita internet)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo", "mongomock"], check=True)
import mongomock

con = sqlite3.connect(ruta_bd)
con.row_factory = sqlite3.Row
docs = [dict(f) for f in con.execute("SELECT id_reserva AS _id, canal, noches, precio_noche, cancelada FROM reservas LIMIT 200")]
con.close()

cliente = mongomock.MongoClient()
coleccion = cliente["turisdata"]["reservas_muestra"]
coleccion.insert_many(docs)
print(coleccion.count_documents({}), coleccion.find_one({}))

### Ejercicio 6 · CRUD en MongoDB
Sobre `coleccion` (200 documentos):
1. **Crear:** inserta un documento `{"_id": 900001, "canal": "directo", "noches": 3, "precio_noche": 99.0, "cancelada": 0}`.
2. **Actualizar:** pon `precio_noche` en `105.0` a ese documento con `$set`.
3. **Leer:** guarda en `precio_leido` su precio leyéndolo de la colección.
4. **Borrar:** elimina todos los documentos con `cancelada` igual a 1 y guarda en `n_borrados` el número (`deleted_count`).
5. Guarda en `n_final` los documentos que quedan.

In [ ]:
# TODO: insert_one, update_one con {"$set": {...}}, find_one, delete_many({"cancelada": 1}).deleted_count y count_documents({})
...
assert precio_leido == 105.0, f"El precio debe quedar en 105.0 y tienes {precio_leido}"
assert n_borrados > 0, "Debe haber documentos con cancelada = 1 en la muestra"
assert n_final == 201 - n_borrados, "Deben quedar 201 (200 + el nuevo) menos los borrados"
assert coleccion.count_documents({"cancelada": 1}) == 0, "No debe quedar ninguna cancelada"
print(f"Correcto: {n_borrados} borrados, quedan {n_final}")

## 7 · Mini-reto integrador: «reservas por canal» desde SQL y desde MongoDB
Escribe una función `contar_por_canal_sql(ruta_bd)` que devuelva un diccionario `{canal: nº de reservas}` **leyendo la base
SQLite** (`sqlite3` + `GROUP BY`). Comprueba que coincide con el resultado de un `count_documents` por canal sobre una
colección de MongoDB cargada con **todas** las reservas. Guarda ambos resultados en `por_canal_sql` y `por_canal_mongo`.

In [ ]:
# TODO: SELECT canal, COUNT(*) ... GROUP BY canal en SQLite; en Mongo, cargar todas las reservas y count_documents({"canal": c}) por cada canal
...
assert por_canal_sql == {"agencia": 1190, "directo": 1462, "empresa": 328, "web_intermediaria": 3020}, f"Recuentos SQL inesperados: {por_canal_sql}"
assert por_canal_sql == por_canal_mongo, "SQL y MongoDB deben dar los mismos recuentos"
print("Correcto:", por_canal_sql)

## 8 · Para reflexionar
1. ¿Qué pasa si construyes una consulta pegando texto del usuario en el SQL? Da un ejemplo de qué podría escribir un atacante.
2. ¿Por qué es importante `commit`? ¿Qué pasaría si el programa se interrumpe antes de hacerlo?
3. ¿Cuándo cargarías un CSV en una base de datos en lugar de leerlo siempre con pandas?

## 9 · Qué has aprendido
- `sqlite3`: `connect`, `execute`, `executemany`, `fetchone/fetchall`, `commit`, `rollback`, `rowcount`.
- Consultas parametrizadas con `?` (protección contra la inyección SQL).
- `read_sql_query` y `to_sql` para pasar de la base a pandas y de pandas a la base.
- CRUD en MongoDB con `pymongo`/`mongomock` y su equivalencia con SQL.
- Ya tienes la **base poblada** (`ocupacion_diaria` dentro de la copia de `turisdata.db`) para el reto.

> **Nota para el reto:** en el reto tendrás que repetir el paso 3 con **tu propia base** (un archivo `.db` que entregarás).
> También existe **SQLAlchemy** para trabajar con distintas bases de datos con el mismo código; no lo usamos aquí para no depender de más librerías.